<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

This notebook contains the executable calibration workflow. Problem definition, mathematical theory, and design rationale are maintained in the companion notebooks.

## Setup — Environment and Configuration

Imports, calibration constants, and the output directory used by all 13 required tasks.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Keep numerical output compact while retaining useful precision.
np.set_printoptions(precision=6, suppress=True)

In [ ]:
SQUARE_SIZE_M = 0.03
INTERNAL_CORNERS_X = 8
INTERNAL_CORNERS_Y = 6
MIN_VALID_VIEWS = 3

In [ ]:
# Directory used to store generated figures.
OUTPUT_DIR = Path("../outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the Sorted JPEG Calibration Images

Discover the repository-relative calibration folder, load the `.jpg` filenames in deterministic sorted order, and fail explicitly if the input directory or images are missing.

In [ ]:
# The notebook is stored in Camera_Calibration/notebooks/.
DATA_DIR = Path("../data/calibration_images")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Calibration image directory not found: {DATA_DIR}"
    )

# Process calibration images in filename order for reproducibility.
image_paths = sorted(DATA_DIR.glob("*.jpg"))

if not image_paths:
    raise FileNotFoundError(
        f"No .jpg calibration images found in: {DATA_DIR}"
    )

print(f"Calibration images found: {len(image_paths)}")
for path in image_paths:
    print(f"  - {path.name}")

## 2. Detect and Refine Chessboard Corners

Implemented by `Image.locate_landmark()` using `cv2.findChessboardCorners` followed by sub-pixel refinement with `cv2.cornerSubPix`.

## 3. Build the Planar World Coordinates

Implemented by `Image.get_landmark_world_coordinate()`: $8 \times 6$ internal corners, $0.03\,\mathrm{m}$ spacing, with the first corner at the planar origin.

## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

`normalize_trans()` translates each 2D point set to zero centroid and scales its mean distance to $\sqrt{2}$.

In [ ]:
def homogenize(points):
    """Convert an array of 2D Cartesian points to homogeneous coordinates."""
    points = np.asarray(points, dtype=float)

    homogeneous = np.ones((points.shape[0], 3), dtype=float)
    homogeneous[:, :2] = points
    return homogeneous


def normalize_trans(points):
    """Return a similarity transform that normalizes a set of 2D points."""
    points = np.asarray(points, dtype=float)

    # Translate the centroid of the point cloud to the origin.
    centroid = np.mean(points, axis=0)
    shifted = points - centroid

    # Scale the mean distance from the origin to sqrt(2).
    distances = np.linalg.norm(shifted, axis=1)
    mean_distance = np.mean(distances)

    if mean_distance < 1e-12:
        raise ValueError(
            "Point normalization is undefined for coincident points."
        )

    scale = np.sqrt(2.0) / mean_distance

    return np.array(
        [
            [scale, 0.0, -scale * centroid[0]],
            [0.0, scale, -scale * centroid[1]],
            [0.0, 0.0, 1.0],
        ],
        dtype=float,
    )

## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

Each correspondence contributes two rows to $Q$. The last right singular vector gives the normalized homography parameters.

## 6. Denormalize Each Homography

The implementation computes

$$
H=T_{\mathrm{image}}^{-1}H_nT_{\mathrm{plane}}
$$

and normalizes the result so that $H_{33}=1$.

In [ ]:
class Image:
    """Represent one planar calibration image and its geometric quantities."""

    # Termination criterion used by OpenCV for sub-pixel corner refinement.
    refine_criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        30,
        0.001,
    )

    def __init__(
        self,
        image_path,
        square_size=0.03,
        rows=8,
        cols=6,
    ):
        self.path = Path(image_path)
        self.square_size = float(square_size)
        self.rows = int(rows)
        self.cols = int(cols)
        self.im_name = self.path.stem

        # OpenCV loads colour images in BGR order.
        self.im = cv2.imread(str(self.path))

        if self.im is None:
            raise FileNotFoundError(
                f"Unable to read calibration image: {self.path}"
            )

        # Compute all geometric quantities associated with this view.
        self.im_pts = self.locate_landmark()
        self.plane_pts = self.get_landmark_world_coordinate()
        self.H = self.find_homography()

    # TASK 2 — Detect and refine chessboard corners.\n    def locate_landmark(self):
        """Detect and refine the internal chessboard corners."""
        gray = cv2.cvtColor(self.im, cv2.COLOR_BGR2GRAY)

        found, corners = cv2.findChessboardCorners(
            gray,
            (self.rows, self.cols),
            None,
        )

        if not found or corners is None:
            raise ValueError(
                f"Chessboard corners were not detected in {self.path.name}"
            )

        # Improve corner localization from pixel to sub-pixel precision.
        corners = cv2.cornerSubPix(
            gray,
            corners,
            (11, 11),
            (-1, -1),
            self.refine_criteria,
        )

        return corners.reshape(-1, 2)

    # TASK 3 — Build planar world coordinates.\n    def get_landmark_world_coordinate(self):
        """Return planar chessboard coordinates in metres."""
        points = np.zeros(
            (self.rows * self.cols, 2),
            dtype=np.float64,
        )

        index = 0

        # The first corner is the planar origin.
        # X increases along the 8-corner direction and Y along the 6-corner direction.
        for y_index in range(self.cols):
            for x_index in range(self.rows):
                points[index, 0] = x_index * self.square_size
                points[index, 1] = y_index * self.square_size
                index += 1

        return points

    # TASKS 4–6 — Normalize points, solve normalized DLT, and denormalize H.\n    def find_homography(self):
        """Estimate the plane-to-image homography using normalized DLT."""
        # TASK 4 — Compute T_image and T_plane.\n        T_image = normalize_trans(self.im_pts)
        T_plane = normalize_trans(self.plane_pts)

        # Transform both point sets into normalized coordinates.
        normalized_image = (
            T_image @ homogenize(self.im_pts).T
        ).T

        normalized_plane = (
            T_plane @ homogenize(self.plane_pts).T
        ).T

        # TASK 5 — Build Q and solve Qh = 0 by SVD.\n        n_points = normalized_image.shape[0]
        Q = np.zeros((2 * n_points, 9), dtype=float)

        # Each point correspondence contributes two linear equations.
        for i in range(n_points):
            X, Y, _ = normalized_plane[i]
            u, v, _ = normalized_image[i]

            Q[2 * i] = [
                X, Y, 1.0,
                0.0, 0.0, 0.0,
                -u * X, -u * Y, -u,
            ]

            Q[2 * i + 1] = [
                0.0, 0.0, 0.0,
                X, Y, 1.0,
                -v * X, -v * Y, -v,
            ]

        # Solve Qh = 0. The last right singular vector gives the
        # least-squares homogeneous solution for the homography.
        _, _, Vt = np.linalg.svd(Q)
        H_normalized = Vt[-1].reshape(3, 3)

        # TASK 6 — Denormalize the homography.\n        # Undo the point normalizations.
        H = (
            np.linalg.inv(T_image)
            @ H_normalized
            @ T_plane
        )

        if abs(H[2, 2]) < 1e-12:
            raise ValueError(
                f"Degenerate homography for {self.path.name}"
            )

        # Homographies are defined up to an arbitrary non-zero scale.
        H /= H[2, 2]
        return H

    @staticmethod
    def _v_ij(hi, hj):
        """Construct one six-element Zhang constraint vector."""
        return np.array(
            [
                hi[0] * hj[0],
                hi[0] * hj[1] + hi[1] * hj[0],
                hi[1] * hj[1],
                hi[2] * hj[0] + hi[0] * hj[2],
                hi[2] * hj[1] + hi[1] * hj[2],
                hi[2] * hj[2],
            ],
            dtype=float,
        )

    def construct_v(self):
        """Return the two independent intrinsic constraints for this homography."""
        h1 = self.H[:, 0]
        h2 = self.H[:, 1]

        v12 = self._v_ij(h1, h2)
        v11 = self._v_ij(h1, h1)
        v22 = self._v_ij(h2, h2)

        return np.vstack((v12, v11 - v22))

    def find_extrinsic(self, K):
        """Recover rotation R and translation t from K and the homography."""
        K_inv = np.linalg.inv(K)

        h1 = self.H[:, 0]
        h2 = self.H[:, 1]
        h3 = self.H[:, 2]

        # Use the first homography column to recover the common scale.
        scale = 1.0 / np.linalg.norm(K_inv @ h1)

        r1 = scale * (K_inv @ h1)
        r2 = scale * (K_inv @ h2)
        r3 = np.cross(r1, r2)

        # Numerical estimation does not guarantee a perfectly orthonormal rotation.
        # Project the approximate matrix onto the nearest valid rotation matrix.
        R_approx = np.column_stack((r1, r2, r3))
        U, _, Vt = np.linalg.svd(R_approx)
        R = U @ Vt

        # Enforce a proper rotation with determinant +1.
        if np.linalg.det(R) < 0:
            U[:, -1] *= -1
            R = U @ Vt

        t = scale * (K_inv @ h3)

        return R, t

### Apply Tasks 1–6 to All Calibration Views

Instantiate one `Image` object per input file, skip invalid chessboard detections, and require at least three valid views.

In [ ]:
images = []

for image_path in image_paths:
    try:
        image = Image(
            image_path,
            square_size=SQUARE_SIZE_M,
            rows=INTERNAL_CORNERS_X,
            cols=INTERNAL_CORNERS_Y,
        )
        images.append(image)
        print(f"[OK]      {image_path.name}")

    except (FileNotFoundError, ValueError) as error:
        print(f"[SKIPPED] {image_path.name}: {error}")

# Zhang calibration requires multiple independent planar views.
if len(images) < MIN_VALID_VIEWS:
    raise RuntimeError(
        f"At least {MIN_VALID_VIEWS} valid calibration images are required."
    )

print(f"\nValid calibration views: {len(images)}")

### Intermediate Data Validation

Verify that every retained view contains the expected 48 image/plane points and a finite homography.

In [ ]:
expected_points = INTERNAL_CORNERS_X * INTERNAL_CORNERS_Y

for image in images:
    assert image.im_pts.shape == (expected_points, 2)
    assert image.plane_pts.shape == (expected_points, 2)
    assert np.all(np.isfinite(image.H))

print("Data validation passed.")

## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

Each valid homography contributes two Zhang constraints. All constraints are stacked into $V$ and solved using the last right singular vector.

In [ ]:
# Stack the two intrinsic constraints from every calibration view.
V = np.vstack([image.construct_v() for image in images])

# Solve Vb = 0 with SVD.
# The right singular vector associated with the smallest singular value
# gives the homogeneous least-squares solution.
_, singular_values, Vt = np.linalg.svd(V)
b = Vt[-1]

print("Constraint matrix shape:", V.shape)
print("\nSingular values:")
print(singular_values)

print("\nResidual norm ||Vb||:")
print(np.linalg.norm(V @ b))

print("\nEstimated b vector:")
print(b)

## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

Recover the five intrinsic parameters from $b$, handle the homogeneous sign ambiguity when required, and assemble the intrinsic matrix $K$.

In [ ]:
b11, b12, b22, b13, b23, b33 = b

denominator = b11 * b22 - b12**2

if abs(denominator) < 1e-12:
    raise ValueError("Degenerate intrinsic calibration constraints.")

# Recover the vertical principal-point coordinate.
v0 = (b12 * b13 - b11 * b23) / denominator

# Scale factor used by the closed-form intrinsic solution.
lambda_ = b33 - (
    b13**2
    + v0 * (b12 * b13 - b11 * b23)
) / b11

# The homogeneous vector b is defined up to sign.
# Reverse it if needed so that the square-root terms remain physically valid.
if lambda_ / b11 <= 0 or lambda_ * b11 / denominator <= 0:
    b = -b
    b11, b12, b22, b13, b23, b33 = b

    denominator = b11 * b22 - b12**2
    v0 = (b12 * b13 - b11 * b23) / denominator

    lambda_ = b33 - (
        b13**2
        + v0 * (b12 * b13 - b11 * b23)
    ) / b11

alpha = np.sqrt(lambda_ / b11)
beta = np.sqrt(lambda_ * b11 / denominator)

gamma = -b12 * alpha**2 * beta / lambda_
u0 = gamma * v0 / beta - b13 * alpha**2 / lambda_

K = np.array(
    [
        [alpha, gamma, u0],
        [0.0, beta, v0],
        [0.0, 0.0, 1.0],
    ],
    dtype=float,
)

print("Intrinsic parameters")
print("--------------------")
print(f"alpha : {alpha:.6f}")
print(f"beta  : {beta:.6f}")
print(f"gamma : {gamma:.6f}")
print(f"u0    : {u0:.6f}")
print(f"v0    : {v0:.6f}")

print("\nIntrinsic matrix K:")
print(K)

## 9. Recover $R$ and $t$ for Every Retained View

Use `Image.find_extrinsic(K)` to recover the pose associated with each valid homography.

In [ ]:
# Recover one pose (R, t) for each retained calibration view.
pose_results = []

for image in images:
    R, t = image.find_extrinsic(K)

    pose_results.append(
        {
            "name": image.path.name,
            "R": R,
            "t": t,
        }
    )

for result in pose_results:
    print(result["name"])
    print("R =")
    print(result["R"])
    print("t =", result["t"])
    print("-" * 60)


## 10. Reproject the $Z=0$ Calibration Points

Transform the planar points into the camera frame, apply $K$, and convert homogeneous image coordinates back to Cartesian pixel coordinates.

In [ ]:
# Reproject every planar calibration point using the recovered camera model.
calibration_results = []

for image, pose in zip(images, pose_results):
    R = pose["R"]
    t = pose["t"]

    # Embed planar (X, Y) coordinates in 3D with Z = 0.
    world_xy = image.get_landmark_world_coordinate()
    world_xyz = np.column_stack(
        (
            world_xy,
            np.zeros(world_xy.shape[0]),
        )
    )

    # World frame -> camera frame.
    camera_points = world_xyz @ R.T + t

    # Camera frame -> homogeneous image coordinates -> Cartesian pixels.
    projected_h = camera_points @ K.T
    projected_pixels = (
        projected_h[:, :2]
        / projected_h[:, 2:3]
    )

    calibration_results.append(
        {
            "name": image.path.name,
            "R": R,
            "t": t,
            "projected": projected_pixels,
        }
    )

print(f"Reprojected calibration views: {len(calibration_results)}")


## 11. Compute Point-wise Errors, Mean Error and RMSE

Evaluate image-space geometric consistency for every calibration point, each view, and the complete dataset.

In [ ]:
# Compute point-wise Euclidean reprojection errors, then mean error and RMSE.
for image, result in zip(images, calibration_results):
    residuals = result["projected"] - image.im_pts
    errors = np.linalg.norm(residuals, axis=1)

    result["errors"] = errors
    result["mean_error"] = float(errors.mean())
    result["rmse"] = float(np.sqrt(np.mean(errors**2)))

for result in calibration_results:
    print(result["name"])
    print(f"Mean reprojection error : {result['mean_error']:.4f} px")
    print(f"Reprojection RMSE       : {result['rmse']:.4f} px")
    print("-" * 60)


In [ ]:
# Combine residual magnitudes from every valid calibration image.
all_errors = np.concatenate(
    [result["errors"] for result in calibration_results]
)

overall_mean_error = float(all_errors.mean())
overall_rmse = float(np.sqrt(np.mean(all_errors**2)))

print("Calibration summary")
print("-------------------")
print(f"Valid calibration views : {len(images)}")
print(f"Calibration points       : {len(all_errors)}")
print(f"Mean reprojection error  : {overall_mean_error:.4f} px")
print(f"Overall RMSE             : {overall_rmse:.4f} px")

## 12. Produce and Save the Six Required Diagnostic Figures

All figures are written to `../outputs/figures/`.

### 12.1 Homography Estimation Pipeline

In [ ]:
# Summarize the normalized DLT homography estimation pipeline.
fig, ax = plt.subplots(figsize=(15, 3.5))
ax.axis("off")

labels = [
    "World points\n(X, Y, 1)",
    "Image points\n(u, v, 1)",
    "Normalize\nT_plane, T_image",
    "Build Q\nQh = 0",
    "SVD\nSolve for h",
    "Denormalize\nH",
]

x_positions = np.linspace(0.08, 0.92, len(labels))

for x, label in zip(x_positions, labels):
    ax.text(
        x,
        0.5,
        label,
        ha="center",
        va="center",
        fontsize=12,
        transform=ax.transAxes,
        bbox={
            "boxstyle": "round,pad=0.5",
            "fill": False,
            "linewidth": 1.5,
        },
    )

for start, end in zip(x_positions[:-1], x_positions[1:]):
    ax.annotate(
        "",
        xy=(end - 0.055, 0.5),
        xytext=(start + 0.055, 0.5),
        xycoords=ax.transAxes,
        arrowprops={
            "arrowstyle": "->",
            "linewidth": 1.5,
        },
    )

ax.set_title("Homography Estimation Pipeline", fontsize=16, pad=20)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "homography_estimation_pipeline.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.2 Mean Reprojection Error by View

In [ ]:
# Compare mean reprojection error across calibration views.
image_names = [
    result["name"]
    for result in calibration_results
]

mean_errors = [
    result["mean_error"]
    for result in calibration_results
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    image_names,
    mean_errors,
)

# The global mean provides a common reference across views.
ax.axhline(
    overall_mean_error,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall mean = {overall_mean_error:.3f} px",
)

for bar, value in zip(bars, mean_errors):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title("Mean Reprojection Error by Calibration View")
ax.set_xlabel("Calibration image")
ax.set_ylabel("Mean reprojection error (pixels)")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "mean_reprojection_error_by_view.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.3 Reprojection Error Distribution

In [ ]:
# Analyse the distribution of point-wise reprojection errors.
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    all_errors,
    bins=20,
    edgecolor="black",
    alpha=0.8,
)

ax.axvline(
    overall_mean_error,
    linestyle="--",
    linewidth=1.5,
    label=f"Mean = {overall_mean_error:.3f} px",
)

ax.axvline(
    overall_rmse,
    linestyle=":",
    linewidth=1.5,
    label=f"RMSE = {overall_rmse:.3f} px",
)

ax.set_title("Distribution of Reprojection Errors")
ax.set_xlabel("Reprojection error (pixels)")
ax.set_ylabel("Number of calibration points")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "reprojection_error_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.4 Detected Chessboard Corners

In [ ]:
# Display detected chessboard corners for all valid calibration views.
n_images = len(images)
n_cols = 3
n_rows = int(np.ceil(n_images / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 5 * n_rows),
)

# Flatten axes to simplify iteration.
axes = np.asarray(axes).reshape(-1)

for ax, image in zip(axes, images):
    visual = image.im.copy()
    corners = image.im_pts.reshape(-1, 1, 2).astype(np.float32)

    # Draw the detected internal chessboard corners.
    cv2.drawChessboardCorners(
        visual,
        (image.rows, image.cols),
        corners,
        True,
    )

    # Convert OpenCV BGR format to RGB for Matplotlib.
    visual_rgb = cv2.cvtColor(
        visual,
        cv2.COLOR_BGR2RGB,
    )

    ax.imshow(visual_rgb)
    ax.set_title(image.path.name)
    ax.axis("off")

# Hide any unused axes in the last row.
for ax in axes[n_images:]:
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "detected_chessboard_corners.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.5 Estimated Camera Poses

In [ ]:
# Visualize the estimated camera centres relative to the chessboard plane.
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# Plot chessboard reference points on the Z = 0 plane.
reference_xy = images[0].plane_pts
reference_xyz = np.column_stack(
    (
        reference_xy,
        np.zeros(reference_xy.shape[0]),
    )
)

ax.scatter(
    reference_xyz[:, 0],
    reference_xyz[:, 1],
    reference_xyz[:, 2],
    marker="s",
    label="Chessboard",
)

for image, result in zip(images, calibration_results):
    R = result["R"]
    t = result["t"]

    # For X_camera = R X_world + t, the camera centre in world
    # coordinates is C = -R^T t.
    camera_center = -R.T @ t

    ax.scatter(
        camera_center[0],
        camera_center[1],
        camera_center[2],
        marker="o",
    )

    ax.text(
        camera_center[0],
        camera_center[1],
        camera_center[2],
        image.path.stem,
        fontsize=8,
    )

ax.set_title("Estimated Camera Poses")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "estimated_camera_poses.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.6 Detected vs Reprojected Points

In [ ]:
# Display detected and reprojected corners for all calibration views.
n_images = len(images)
n_cols = 3
n_rows = int(np.ceil(n_images / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 5 * n_rows),
)

axes = np.asarray(axes).reshape(-1)

for ax, (image, result) in zip(
    axes,
    zip(images, calibration_results),
):
    image_rgb = cv2.cvtColor(
        image.im,
        cv2.COLOR_BGR2RGB,
    )

    ax.imshow(image_rgb)

    # Observed corner positions.
    ax.scatter(
        image.im_pts[:, 0],
        image.im_pts[:, 1],
        s=24,
        marker="o",
        label="Detected",
    )

    # Corner positions predicted by the estimated camera model.
    ax.scatter(
        result["projected"][:, 0],
        result["projected"][:, 1],
        s=24,
        marker="x",
        label="Reprojected",
    )

    ax.set_title(
        f"{image.path.name} — RMSE: {result['rmse']:.3f} px"
    )
    ax.axis("off")
    ax.legend(fontsize=8)

for ax in axes[n_images:]:
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "reprojection_results.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 13. Run the Numerical and Output-file Validation Checks

Validate $K$, rotation orthonormality and determinant, finite residuals, and the existence of all six required output figures.

In [ ]:
if K.shape != (3, 3) or not np.all(np.isfinite(K)):
    raise ValueError("Invalid intrinsic matrix K.")

for result in calibration_results:
    R = result["R"]
    if not np.allclose(R.T @ R, np.eye(3), atol=1e-6):
        raise ValueError(f"Non-orthonormal rotation for {result['name']}.")
    if not np.isclose(np.linalg.det(R), 1.0, atol=1e-6):
        raise ValueError(f"Invalid rotation determinant for {result['name']}.")
    if not np.all(np.isfinite(result["errors"])):
        raise ValueError(f"Non-finite residuals for {result['name']}.")

required_outputs = [
    "detected_chessboard_corners.png",
    "homography_estimation_pipeline.png",
    "estimated_camera_poses.png",
    "reprojection_results.png",
    "mean_reprojection_error_by_view.png",
    "reprojection_error_distribution.png",
]
missing = [name for name in required_outputs if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError("Missing outputs: " + ", ".join(missing))

print("All Camera Calibration validation checks passed.")

## Final Result Summary

The pipeline estimates camera intrinsics from multiple planar chessboard views, recovers one pose per valid image, and evaluates geometric consistency through reprojection. Lens-distortion estimation is intentionally outside the scope of this implementation.